In [1]:
import pandas as pd
import pickle as pkl
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=True)
tqdm.pandas()

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [4]:
gene2vec_emb_mat = pd.read_hdf("/work/magroup/kaileyhu/res/perturbed/gf_12L_30M_i2048_SL/gene2vec_df/gene2vec_emb_mat.h5", "table")

In [15]:
sub_embs = pd.read_csv("/work/magroup/kaileyhu/res/perturbed/gf_12L_30M_i2048_SL/generated_df_2026/sub_embs_only.csv")

In [16]:
sub_embs.to_hdf("/work/magroup/kaileyhu/res/perturbed/gf_12L_30M_i2048_SL/generated_df_2026/sub_embs_only.hdf", key="table")

In [17]:
sub_embs

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,503,504,505,506,507,508,509,510,511,512
0,"('ACH-000001', 'gene_ENSG00000051341')",-0.299434,0.001204,0.012678,-0.006565,0.001230,-0.000863,0.003281,-0.001110,-0.015870,...,0.002323,0.000880,-0.006126,-0.006907,0.004198,0.000945,-0.007539,0.011587,-0.001489,-0.004256
1,"('ACH-000004', 'gene_ENSG00000051341')",-0.907835,0.007782,0.006376,-0.000449,0.000370,0.007412,-0.008123,0.000305,-0.000180,...,-0.001291,0.015593,-0.001039,-0.003408,0.003154,0.006301,0.002561,-0.003761,-0.000219,-0.008459
2,"('ACH-000005', 'gene_ENSG00000051341')",0.014349,0.000037,-0.001843,0.002467,0.000757,0.008551,0.005701,-0.001113,0.005304,...,0.000381,0.002028,-0.000999,0.009511,0.000325,-0.004504,0.000199,-0.002697,0.005191,0.007314
3,"('ACH-000007', 'gene_ENSG00000051341')",-0.360821,-0.007642,-0.008493,-0.014396,-0.001038,-0.008241,-0.001631,-0.011372,-0.008973,...,-0.022026,-0.020298,-0.002187,0.006115,0.012386,-0.011641,-0.013912,0.002308,-0.000791,0.001693
4,"('ACH-000009', 'gene_ENSG00000051341')",-0.405454,0.000219,0.006562,-0.003238,-0.000372,-0.007594,0.007154,-0.001102,-0.002828,...,-0.016940,0.003063,-0.005743,-0.010180,-0.004058,0.006763,0.004595,-0.002552,-0.000998,0.012430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2289234,"('ACH-002834', 'gene_ENSG00000100926')",-0.223258,0.000908,0.002944,-0.001630,-0.001210,0.003533,0.008711,0.006551,0.007152,...,-0.005847,-0.000975,-0.003746,0.004230,0.005619,0.003707,0.005144,0.005153,0.007853,-0.000552
2289235,"('ACH-002847', 'gene_ENSG00000100926')",-0.339084,0.000367,-0.000251,-0.006364,-0.002435,0.006657,-0.002796,0.000758,-0.002401,...,0.000757,0.018713,0.002248,0.005138,0.008132,-0.005671,-0.006674,0.001999,-0.000542,-0.005011
2289236,"('ACH-002922', 'gene_ENSG00000100926')",-0.200627,0.001408,0.007166,0.000597,-0.002361,0.004204,-0.013206,-0.001740,-0.018421,...,0.018214,0.029569,0.010049,-0.009038,-0.009370,0.011491,0.005710,-0.020833,0.016583,-0.010284
2289237,"('ACH-002925', 'gene_ENSG00000100926')",-0.298566,0.004282,-0.001453,-0.010616,-0.002173,0.005936,-0.012194,0.004332,-0.009797,...,0.005055,0.029445,0.017018,0.012130,-0.003045,-0.002445,-0.005047,-0.005891,-0.006507,-0.007699


In [20]:
from ast import literal_eval

In [23]:
sub_embs.set_index("Unnamed: 0", inplace = True)

In [24]:
sub_embs['gene'] = list(map(lambda x : literal_eval(x)[1], list(sub_embs.index)))

In [25]:
sub_embs['gene'] = sub_embs['gene'].apply(lambda x : x[5:])

In [26]:
valid_ensembl = set(sub_embs['gene'])

In [27]:
len(valid_ensembl)

6072

In [28]:
SL_df = pd.read_csv("/work/magroup/kaileyhu/datasets/SynLethDB/Human_SL.csv")
nonSL_df = pd.read_csv("/work/magroup/kaileyhu/datasets/SynLethDB/Human_nonSL.csv")

In [29]:
# Pull all pos / neg pairs. If a pair is present in both, delete it

res_pairs = {}
for i in range(len(SL_df)):
    row = SL_df.iloc[i]
    g1 = row["n1.name"]
    g2 = row["n2.name"]
    res_pairs[(g1, g2)] = True

num_overlap = 0
for i in range(len(nonSL_df)):
    row = nonSL_df.iloc[i]
    g1 = row["n1.name"]
    g2 = row["n2.name"]
    if (g1, g2) in res_pairs:
        num_overlap += 1
        del res_pairs[(g1, g2)]
    else:
        res_pairs[(g1, g2)] = False

In [ ]:
with open ("/work/magroup/kaileyhu/datasets/SynLethDB/all_pairs_dict_FIXED_2026.pkl", "wb") as f:
    pkl.dump(res_pairs, f)

In [35]:
with open ("/work/magroup/kaileyhu/datasets/SynLethDB/all_pairs_dict_FIXED_2026.pkl", "rb") as f:
    res_pairs = pkl.load(f)

In [36]:
pair_list = res_pairs

### negative sampling

In [4]:
df = pd.read_csv("/work/magroup/kaileyhu/datasets/depmap/OmicsExpressionProteinCodingGenesTPMLogp1.csv")

In [ ]:
df.set_index("Unnamed: 0", inplace = True)
corr_mat = df.corr()

In [ ]:
corr_mat.to_csv("/work/magroup/kaileyhu/datasets/depmap/NSM_EXP.csv")

In [4]:
corr_mat = pd.read_csv("/work/magroup/kaileyhu/datasets/depmap/NSM_EXP.csv")

In [5]:
corr_mat.set_index("Unnamed: 0", inplace = True)
corr_mat.index = list(map(lambda x : x.split(' ')[0], corr_mat.index))
corr_mat.columns = list(map(lambda x : x.split(' ')[0], corr_mat.columns))
corr_dict = corr_mat.to_dict()

In [ ]:
res = set()
for (k, v) in tqdm(corr_dict.items()):
    for (k2, v2) in v.items():
        if k < k2:
            res.add((k, k2, v2))
        else:
            res.add((k2, k, v2))

sorted_items = sorted(res, key=lambda item: item[2])

In [ ]:
with open ("/work/magroup/kaileyhu/datasets/depmap/sampling/nsm_exp_sorted.pkl", "wb") as f:
    pkl.dump(sorted_items, f)

In [30]:
with open ("/work/magroup/kaileyhu/datasets/depmap/sampling/nsm_exp_sorted.pkl", "rb") as f:
    sorted_items = pkl.load(f)

In [33]:
ensembl_path = "/work/magroup/kaileyhu/Geneformer/geneformer/ensembl_mapping_dict_gc95M.pkl"

def invert_dict(dict_obj):
    return {v: k for k, v in dict_obj.items()}

with open(ensembl_path, "rb") as f:
    id_gene_dict = pkl.load(f)

def query_id(g):
    if g in id_gene_dict:
        if id_gene_dict[g] in valid_ensembl:
            return id_gene_dict[g]
    return " "

In [37]:
# subset pairlist to only valid genes
pair_list = {(g1, g2) : v for ((g1, g2), v) in pair_list.items() if query_id(g1) != " " and query_id(g2) != " "}

In [39]:
len(pair_list)

24189

In [40]:
pos_pairs = {key: value for key, value in pair_list.items() if value}

In [41]:
num_pos = len(pos_pairs)
num_neg = len(pair_list) - num_pos

In [42]:
(len(pair_list), num_pos, num_neg)

(24189, 22081, 2108)

In [49]:
sorted_items_f = list(filter(lambda x :  query_id(x[0]) != " " and  query_id(x[1]) != " ", sorted_items))

In [ ]:
includes_all = []
num_added = 0
sample_param = 5

for i, (g1, g2, corr) in tqdm(enumerate(sorted_items_f)):
    g1_ens, g2_ens = query_id(g1), query_id(g2)
    if g1_ens == " " or g2_ens == " ":
        continue
    if (g1, g2, corr) in includes_all or (g1, g2) in pair_list:
        continue
    if len(includes_all) + num_neg >= sample_param * num_pos:
        break
    includes_all.append((g1, g2, corr))
    num_added += 1
    if (num_added % 10000 == 0):
        print(f"added {num_added} so far")

0it [00:00, ?it/s]

added 10000 so far
added 20000 so far
added 30000 so far
added 40000 so far
added 50000 so far
added 60000 so far
added 70000 so far
added 80000 so far
added 90000 so far
added 100000 so far


In [56]:
for (g1, g2, _) in includes_all:
    pair_list[(g1, g2)] = False

In [57]:
# total, num pos, num neg
len(pair_list), len({key: value for key, value in pair_list.items() if value}), len({key: value for key, value in pair_list.items() if not value})

(152459, 22081, 130378)

In [59]:
with open (f"/work/magroup/kaileyhu/datasets/SynLethSampled/all_pairs_NSM_EXP_{sample_param}x_FIXED_2026_restricted.pkl", "wb") as f:
    pkl.dump(pair_list, f)